<a href="https://colab.research.google.com/github/assen-v/llm-gpu-benchmark/blob/main/01_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
!nvidia-smi


Wed Oct 14 03:11:28 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   56C    P0             27W /   70W |    6401MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [12]:
!pip install -q transformers accelerate bitsandbytes datasets

In [13]:
import torch, time, re
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0))

name = "Qwen/Qwen2.5-1.5B-Instruct"
tok = AutoTokenizer.from_pretrained(name)
tok.padding_side = "left"

2.11.0+cu130 True Tesla T4


In [14]:
ids = tok("What is the capital of France?")["input_ids"]
print(ids)
print(tok.convert_ids_to_tokens(ids))
print(tok.decode(ids))

ids = tok("Quantization of transformers is unbelievably useful")["input_ids"]
print(ids)
print(tok.convert_ids_to_tokens(ids))

[3838, 374, 279, 6722, 315, 9625, 30]
['What', 'Ġis', 'Ġthe', 'Ġcapital', 'Ġof', 'ĠFrance', '?']
What is the capital of France?
[44220, 2022, 315, 86870, 374, 38937, 88134, 5390]
['Quant', 'ization', 'Ġof', 'Ġtransformers', 'Ġis', 'Ġunbelie', 'vably', 'Ġuseful']


In [15]:
model = AutoModelForCausalLM.from_pretrained(name, torch_dtype=torch.float16).to("cuda")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [16]:
!nvidia-smi

Wed Oct 14 03:11:41 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   56C    P0             27W /   70W |    6431MiB /  15360MiB |     32%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [17]:
def bench(model, device, prompts, max_new_tokens=64, runs=3):
    inputs = tok(prompts, return_tensors="pt", padding=True).to(device)
    gen = dict(max_new_tokens=max_new_tokens, min_new_tokens=max_new_tokens, do_sample=False)
    model.generate(**inputs, **{**gen, "max_new_tokens": 8, "min_new_tokens": 8})  # warmup
    if device == "cuda":
        torch.cuda.reset_peak_memory_stats()
    times = []
    for _ in range(runs):
        if device == "cuda":
            torch.cuda.synchronize()
            s, e = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
            s.record()
            out = model.generate(**inputs, **gen)
            e.record(); torch.cuda.synchronize()
            times.append(s.elapsed_time(e) / 1000)
        else:
            t0 = time.perf_counter()
            out = model.generate(**inputs, **gen)
            times.append(time.perf_counter() - t0)
    avg = sum(times) / len(times)
    new_tokens = max_new_tokens * len(prompts)
    mem = torch.cuda.max_memory_allocated() / 1e9 if device == "cuda" else None
    return {"latency_s": avg, "tokens_per_s": new_tokens / avg, "peak_mem_gb": mem}


In [18]:
test_prompts = ["What is the capital of France?"]
model = AutoModelForCausalLM.from_pretrained(name, torch_dtype=torch.float16).to("cuda")
print(bench(model, "cuda", test_prompts, max_new_tokens=32))

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

{'latency_s': 1.2738872884114583, 'tokens_per_s': 25.119961782414915, 'peak_mem_gb': 3.105391104}


In [19]:
print(bench(model, "cuda", test_prompts, max_new_tokens=32))
print(bench(model, "cuda", test_prompts, max_new_tokens=32))

{'latency_s': 1.2688380126953125, 'tokens_per_s': 25.219925380407243, 'peak_mem_gb': 3.105391104}
{'latency_s': 1.2790703531901042, 'tokens_per_s': 25.018170361145053, 'peak_mem_gb': 3.105391104}
